In [10]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import plotly.graph_objects as go
from sklearn.manifold import TSNE

In [11]:
MODEL = "gpt-4.1-nano"
db_name = "vector_db"

# Chunks

In [ ]:
# Each run of notebook 1 has its own dated folder.
# The newest run sorts last because the folder names start with the date.
runs = sorted(glob.glob("knowledge-base/*"))
latest_run = runs[-1]
folders = glob.glob(f"{latest_run}/*")  # The "jobs" and "companies" subfolders
print(f"Loading knowledge base from {latest_run}")

documents = []  # Collects the loaded documents from all folders
for folder in folders:  # Go through each subfolder
    doc_type = os.path.basename(folder)  # "jobs" or "companies"
    loader = DirectoryLoader(  # Create a loader for this folder
        folder,  # Folder to load files from
        glob="**/*.md",  # Only load Markdown files, including in subfolders
        loader_cls=TextLoader,  # Read each file as plain text
        loader_kwargs={"encoding": "utf-8"},  # Read files as UTF-8
    )
    folder_docs = loader.load()  # Load all files into LangChain Document objects
    for doc in folder_docs:  # Go through each loaded document
        doc.metadata["doc_type"] = doc_type  # Tag the document with its type
        documents.append(doc)  # Add the document to the full list

print(f"Loaded {len(documents)} documents")  # Show how many documents were loaded

In [13]:
documents[0]  # Show the first document to verify it was loaded correctly

Document(metadata={'source': 'knowledge-base/2026-09-14_1417/jobs/02-cyrad-solutions-agentic-ai-engineer-scientific-discovery.md', 'doc_type': 'jobs'}, page_content='---\ntitle: "Agentic AI Engineer – Scientific Discovery"\ncompany: "Cyrad Solutions"\ncompany_file: "companies/cyrad-solutions.md"\nlocation: "New York, NY, US"\njob_url: "https://www.indeed.com/viewjob?jk=7fe308e11137e0ba"\nscraped_at: "2026-09-14_1417"\n---\n\n# Agentic AI Engineer – Scientific Discovery\n\n* New York, New York, United States\n\n  \n**Location:** New York, NY  \n\n**Work Model:** Hybrid  \n\n**Compensation:** $250K to $600K base, plus sign\\-on and year\\-end bonuses\n\n\n**Overview**\n\n\n\nA well\\-established scientific research organization is standing up a new AI initiative at the intersection of agentic systems, generative AI, scientific computing, and drug discovery. The work is greenfield: rather than maintaining mature systems, the role will help define how AI agents, retrieval systems, and inte

In [14]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 173 chunks
First chunk:

page_content='---
title: "Agentic AI Engineer – Scientific Discovery"
company: "Cyrad Solutions"
company_file: "companies/cyrad-solutions.md"
location: "New York, NY, US"
job_url: "https://www.indeed.com/viewjob?jk=7fe308e11137e0ba"
scraped_at: "2026-09-14_1417"
---

# Agentic AI Engineer – Scientific Discovery

* New York, New York, United States

  
**Location:** New York, NY  

**Work Model:** Hybrid  

**Compensation:** $250K to $600K base, plus sign\-on and year\-end bonuses


**Overview**



A well\-established scientific research organization is standing up a new AI initiative at the intersection of agentic systems, generative AI, scientific computing, and drug discovery. The work is greenfield: rather than maintaining mature systems, the role will help define how AI agents, retrieval systems, and intelligent workflows are built and deployed for researchers working on hard problems in molecular science.' metadata={'source': 'knowledge-base/2

# Vectors

In [ ]:
# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(
    documents=chunks, embedding=embeddings, persist_directory=db_name
)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 173 documents


In [16]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 173 vectors with 3,072 dimensions in the vector store


In [17]:
# Prework

result = collection.get(include=["embeddings", "documents", "metadatas"])
vectors = np.array(result["embeddings"])
documents = result["documents"]
metadatas = result["metadatas"]
doc_types = [metadata["doc_type"] for metadata in metadatas]

# One color per document type
type_colors = {"jobs": "blue", "companies": "orange"}

In [ ]:

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)
fig = go.Figure()

for doc_type, color in type_colors.items():
    indexes = [i for i, t in enumerate(doc_types) if t == doc_type]
    fig.add_trace(
        go.Scatter(
            x=reduced_vectors[indexes, 0],
            y=reduced_vectors[indexes, 1],
            mode="markers",
            name=doc_type,
            marker=dict(size=5, color=color, opacity=0.8),
            text=[
                f"Type: {doc_type}<br>Text: {documents[i][:100]}..." for i in indexes
            ],
            hoverinfo="text",
        )
    )

fig.update_layout(
    title="2D Chroma Vector Store Visualization",
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40),
)

fig.show()

In [ ]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

fig = go.Figure()

for doc_type, color in type_colors.items():
    indexes = [i for i, t in enumerate(doc_types) if t == doc_type]
    fig.add_trace(
        go.Scatter3d(
            x=reduced_vectors[indexes, 0],
            y=reduced_vectors[indexes, 1],
            z=reduced_vectors[indexes, 2],
            mode="markers",
            name=doc_type,
            marker=dict(size=5, color=color, opacity=0.8),
            text=[
                f"Type: {doc_type}<br>Text: {documents[i][:100]}..." for i in indexes
            ],
            hoverinfo="text",
        )
    )

fig.update_layout(
    title="3D Chroma Vector Store Visualization",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40),
)

fig.show()